In [ ]:
import sys
import os
import json

sys.path.append(os.path.abspath(".."))

import polars as pl
from db import Database
from s3 import S3Client

In [ ]:
db = Database("raw_data")
s3 = S3Client()

In [ ]:
from datetime import date

# -------------------------
# 📅 Date
# -------------------------
run_date = "2026-06-22"


# -------------------------
# 📦 Sources
# -------------------------
sources = [
    "africanandeastern",
    "africaneastern-abu-dabhi",
    "africaneastern-dubai",
    "africaneastern-rak",
    "mmihomedelivery-dubai",
    "mmihomedelivery-rak",
    "bdutyfree",
    "bmmishops",
    "centaurusint",
    "citydrinks",
    "dubaidutyfree",
    "gbiexpress",
    "gmp",
    "highspiritsuae",
    "jmkcellars",
    "nhsc",
    "royalspirit",
    "spinneysauh",
    "supercellar",
    "thebottlestore",
    "unisatgt",
    "windmillgt",
]


# -------------------------
# 🔗 Build URLs
# -------------------------
all_data = []
for source in sources:
    data = s3.read_json(f"s3://raw-data-bucket/beer-pricing/{source}/{run_date}.json")
    print(f"Got {source} ------- {len(data)}")
    all_data.extend(data)

print(f"Got final {len(all_data)}")


In [ ]:
from datetime import date

# -------------------------
# 📅 Date
# -------------------------
run_date = "2026-05-12"


# -------------------------
# 📦 Sources
# -------------------------
sources = [
    # "africaneastern-abu-dabhi"
    "centaurusint",
    "supercellar",
    "unisatgt",
    "windmillgt"
]


# -------------------------
# 🔗 Build URLs
# -------------------------
for source in sources:
    data = s3.read_json(f"s3://raw-data-bucket/beer-pricing/{source}/{run_date}.json")
    print(f"Got {source} ------- {len(data)}")
    all_data.extend(data)

print(f"Got final {len(all_data)}")


In [ ]:
df = pl.DataFrame(all_data)
df

In [ ]:
# update region label
df = df.with_columns(
    pl.when(pl.col("region") == "Northern Emirates")
    .then(pl.lit("Other Emirates"))
    .otherwise(pl.col("region"))
    .alias("region")
)


In [ ]:
# mapping dictionary
mapping_dict = {
    "jmkcellars": "Other Emirates",
    "africanandeastern": "Bahrain",
    "royalspirit": "Abu Dhabi",
    "highspiritsuae": "Abu Dhabi",
    "citydrinks": "Dubai",
    "unisatgt": "Other Emirates",
    "dubaidutyfree": "Dubai",
    "bmmishops": "Bahrain",
    "gbiexpress": "Bahrain",
    "nhsc": "Bahrain",
    "supercellar": "Other Emirates",
    "bdutyfree": "Bahrain",
    "windmillgt": "Abu Dhabi",
    "thebottlestore": "Abu Dhabi",
    "spinneysauh": "Abu Dhabi",
    "centaurusint": ""
}

# add region column
df = df.with_columns(
    pl.col("platform").replace(mapping_dict).alias("region")
)

# normalize region labels
df = df.with_columns(
    pl.when(pl.col("region") == "Northern Emirates")
    .then(pl.lit("Other Emirates"))
    .otherwise(pl.col("region"))
    .alias("region")
)

# optional: quick check
print(df.select("platform", "region").unique())

In [ ]:
df

In [ ]:
df["region"].unique().to_list()

In [ ]:
df["platform"].unique().to_list()

In [ ]:
len(df["platform"].unique().to_list())

In [ ]:
df["brand"].unique().to_list()

In [ ]:
len(df["brand"].unique().to_list())

In [ ]:
df = df.with_columns(
    pl.col("brand")
    .cast(pl.Utf8)
    .str.to_lowercase()
    .str.strip_chars()
    .str.replace_all(r"[^a-z0-9\s]", "")
    .str.replace_all(
        r"\b(strong|light|ipa|weissbier|pils|imported|beer|cider)\b",
        ""
    )
    .str.replace_all(r"\s+", " ")
    .str.strip_chars()
    .alias("cleaned_brand")
)

In [ ]:
len(df["cleaned_brand"].unique().to_list())

In [ ]:
df["cleaned_brand"].unique().to_list()

In [ ]:
import polars as pl

# define your 4 platforms
target_platforms = [
    "africanandeastern",
    "bmmishops",
    "gbiexpress",
    "nhsc",
    "bdutyfree"
]

df = df.with_columns(
    pl.when(pl.col("platform").is_in(target_platforms))
    .then(pl.lit("Baharain"))          # country for those 4
    .otherwise(pl.lit("UAE"))   # remaining
    .alias("country")
)

df

In [ ]:
invalid = [
    "super premium","premium","mainstream",
    "strong","craft / speciality","cider / flavored","other"
]

df = df.with_columns(
    pl.when(pl.col("cleaned_brand").is_in(invalid))
    .then(None)
    .otherwise(pl.col("cleaned_brand"))
    .alias("cleaned_brand")
)

In [ ]:
segment_mapping_updated = {

    # -----------------
    # 🟢 SUPER PREMIUM
    # -----------------
    "almaza":                   "super premium",
    "al maza":                  "super premium",
    "asahi":                    "super premium",
    "birra moretti":            "super premium",
    "peroni":                   "super premium",
    "peroni nastro azzurro":    "super premium",
    "estrella":                 "super premium",
    "estrella damm":            "super premium",
    "estrella galicia":         "super premium",
    "dos equis":                "super premium",
    "hitachino":                "super premium",
    "kirin":                    "super premium",
    "moritz":                   "super premium",
    "alhambra":                 "super premium",
    "mahou":                    "super premium",
    "modelo":                   "super premium",
    "modelo especial":          "super premium",
    "brooklyn":                 "super premium",
    "sierra nevada":            "super premium",
    "goose island":             "super premium",
    "pirate life":              "super premium",
    "kaiju":                    "super premium",
    "engkanto":                 "super premium",
    "quilmes":                  "super premium",
    "saigon":                   "super premium",
    "chang":                    "super premium",
    "mythos":                   "super premium",
    "fix":                      "super premium",
    "cisk":                     "super premium",

    # -----------------
    # 🔵 PREMIUM
    # -----------------
    "corona":                   "premium",
    "heineken":                 "premium",
    "heineken silver":          "premium",
    "stella artois":            "premium",
    "stella":                   "premium",
    "san miguel":               "premium",
    "oranjeboom":               "premium",
    "windhoek":                 "premium",
    "kronenbourg":              "premium",
    "kronenbourg 1664 lager":   "premium",
    "kronenbourg beer":         "premium",
    "michelob":                 "premium",
    "desperados":               "premium",
    "hollandia":                "premium",
    "hofbrau":                  "premium",
    "grolsch":                  "premium",
    "bavaria":                  "premium",
    "schofferhofer":            "premium",
    "maisel":                   "premium",
    "radeberger":               "premium",
    "jever":                    "premium",
    "paderborner":              "premium",
    "starnberger":              "premium",
    "edelhopfen":               "premium",
    "vitalsberg":               "premium",

    # -----------------
    # 🟡 MAINSTREAM
    # -----------------
    "amstel":                   "mainstream",
    "amstel light":             "mainstream",
    "becks":                    "mainstream",
    "beck's":                   "mainstream",
    "budweiser":                "mainstream",
    "carlsberg":                "mainstream",
    "tiger":                    "mainstream",
    "singha":                   "mainstream",
    "sol":                      "mainstream",
    "tuborg":                   "mainstream",
    "holsten":                  "mainstream",
    "efes":                     "mainstream",
    "efes pilsner":             "mainstream",
    "warsteiner":               "mainstream",
    "warsteiner lager":         "mainstream",
    "krombacher":               "mainstream",
    "krombacher lager":         "mainstream",
    "bitburger":                "mainstream",
    "bitburger beer":           "mainstream",
    "baltika":                  "mainstream",
    "baltika lager":            "mainstream",
    "tsingtao":                 "mainstream",
    "dab":                      "mainstream",
    "carling":                  "mainstream",
    "staropramen":              "mainstream",
    "tennents":                 "mainstream",
    "lowenbrau":                "mainstream",
    "löwenbräu":                "mainstream",
    "lwenbru":                  "mainstream",
    "miller":                   "mainstream",
    "okocim":                   "mainstream",
    "poretti":                  "mainstream",
    "angelo poretti":           "mainstream",
    "just life":                "mainstream",
    "zibert":                   "mainstream",
    "budejovicky":              "mainstream",
    "budvar":                   "mainstream",
    "castle":                   "mainstream",
    "coors":                    "mainstream",
    "kilkenny":                 "mainstream",
    "kaiserdom":                "mainstream",
    "matz":                     "mainstream",
    "hollandia":                "mainstream",
    "beirut":                   "mainstream",
    "almaza":                   "mainstream",
    "majlis":                   "mainstream",

    # -----------------
    # 🔴 STRONG
    # -----------------
    "kingfisher":               "strong",
    "king fisher":              "strong",
    "king fisher blue premium": "strong",
    "haywards":                 "strong",
    "red horse":                "strong",
    "kalyani":                  "strong",
    "godfather":                "strong",
    "lion":                     "strong",
    "lion brewery":             "strong",
    "simba":                    "strong",
    "royal dutch":              "strong",
    "mannilla":                 "strong",
    "mannila":                  "strong",
    "manila":                   "strong",
    "cobra":                    "strong",
    "commando":                 "strong",
    "maximus":                  "strong",
    "bira":                     "strong",
    "himalayan":                "strong",
    "barahsinghe":              "strong",
    "8 horse":                  "strong",
    "buffelsfontein":           "strong",
    "loxtonia":                 "strong",

    # -----------------
    # 🟣 CRAFT / SPECIALITY
    # -----------------
    "guinness":                 "craft / speciality",
    "hoegaarden":               "craft / speciality",
    "hoegarden":                "craft / speciality",
    "blue moon":                "craft / speciality",
    "brewdog":                  "craft / speciality",
    "paulaner":                 "craft / speciality",
    "murphys":                  "craft / speciality",
    "murphys irish stout":      "craft / speciality",
    "murphy's":                 "craft / speciality",
    "erdinger":                 "craft / speciality",
    "erdinger wheat beer":      "craft / speciality",
    "franziskaner":             "craft / speciality",
    "weihenstephaner":          "craft / speciality",
    "benediktiner":             "craft / speciality",
    "leffe":                    "craft / speciality",
    "coopers":                  "craft / speciality",
    "goose island":             "craft / speciality",
    "abbot":                    "craft / speciality",
    "abbot ale":                "craft / speciality",
    "crew republic":            "craft / speciality",
    "duvel":                    "craft / speciality",
    "rochehaut":                "craft / speciality",
    "o'hara's":                 "craft / speciality",
    "oharas":                   "craft / speciality",
    "belhaven":                 "craft / speciality",
    "wittekerke":               "craft / speciality",
    "kaiju":                    "craft / speciality",
    "devil's peak":             "craft / speciality",
    "devils peak":              "craft / speciality",
    "petrus":                   "craft / speciality",
    "chimay":                   "craft / speciality",
    "chouffe":                  "craft / speciality",
    "delirium":                 "craft / speciality",
    "westmalle":                "craft / speciality",
    "rochefort":                "craft / speciality",
    "orval":                    "craft / speciality",
    "affligem":                 "craft / speciality",
    "grimbergen":               "craft / speciality",
    "kasteel":                  "craft / speciality",
    "kwak":                     "craft / speciality",
    "pauwel kwak":              "craft / speciality",
    "lindemans":                "craft / speciality",
    "fruli":                    "craft / speciality",
    "la trappe":                "craft / speciality",
    "st. bernardus":            "craft / speciality",
    "st bernardus":             "craft / speciality",
    "gulden draak":             "craft / speciality",
    "piraat":                   "craft / speciality",
    "malheur":                  "craft / speciality",
    "la guillotine":            "craft / speciality",
    "filou":                    "craft / speciality",
    "kristoffel":               "craft / speciality",
    "zundert":                  "craft / speciality",
    "tynt meadow":              "craft / speciality",
    "st. feuillien":            "craft / speciality",
    "st feuillien":             "craft / speciality",
    "maredsous":                "craft / speciality",
    "goudale":                  "craft / speciality",
    "triple karmeliet":         "craft / speciality",
    "texels":                   "craft / speciality",
    "two chefs":                "craft / speciality",
    "moersleutel":              "craft / speciality",
    "jaipur":                   "craft / speciality",
    "thornbridge":              "craft / speciality",
    "hobgoblin":                "craft / speciality",
    "old speckled hen":         "craft / speciality",
    "greene king":              "craft / speciality",
    "belhaven":                 "craft / speciality",
    "the white hag":            "craft / speciality",
    "two tribes":               "craft / speciality",
    "de koninck":               "craft / speciality",
    "liefmans":                 "craft / speciality",
    "bush":                     "craft / speciality",
    "rye river":                "craft / speciality",
    "abbaye st idesbald":       "craft / speciality",
    "henry westons":            "craft / speciality",
    "blackthorn":               "craft / speciality",
    "maeloc":                   "craft / speciality",
    "seven rivers":             "craft / speciality",
    "craft nation":             "craft / speciality",
    "pirate life":              "craft / speciality",
    "sierra nevada":            "craft / speciality",
    "brooklyn":                 "craft / speciality",
    "snowy":                    "craft / speciality",
    "grins":                    "craft / speciality",
    "maisel":                   "craft / speciality",
    "knig ludwig":              "craft / speciality",
    "könig ludwig":             "craft / speciality",
    "starnberger":              "craft / speciality",
    "hofbrau":                  "craft / speciality",
    "hop house":                "craft / speciality",
    "kilkenny":                 "craft / speciality",

    # -----------------
    # 🟠 CIDER / FLAVORED
    # -----------------
    "hunters":                  "cider/ flavored",
    "hunter's":                 "cider/ flavored",
    "hunter":                   "cider/ flavored",
    "hunters gold":             "cider/ flavored",
    "hunters hard lemon":       "cider/ flavored",
    "savanna":                  "cider/ flavored",
    "savanna dry angry lemon":  "cider/ flavored",
    "strongbow":                "cider/ flavored",
    "thatchers":                "cider/ flavored",
    "magners":                  "cider/ flavored",
    "rekorderlig":              "cider/ flavored",
    "weston cider":             "cider/ flavored",
    "somersby":                 "cider/ flavored",
    "crabbies":                 "cider/ flavored",
    "crabbie's":                "cider/ flavored",
    "loxtonia":                 "cider/ flavored",
    "salitos":                  "cider/ flavored",
    "kopparberg":               "cider/ flavored",
    "kiss":                     "cider/ flavored",
    "maeloc":                   "cider/ flavored",

    # -----------------
    # ⚫ NON-ALC
    # -----------------
    "heineken 0.0":             "non alc",

    # -----------------
    # ⚫ OTHER
    # -----------------
    "smirnoff":                 "other",
    "bacardi":                  "other",
    "cutty sark":               "other",
    "manito mezcal":            "other",
    "manito":                   "other",
    "heavensake":               "other",
    "dassai":                   "other",
    "sho chiku bai":            "other",
    "x mark vodka":             "other",
    "x mark":                   "other",
    "xmark":                    "other",
    "x-mark":                   "other",
    "x - mark":                 "other",
    "buzz vodka orange":        "other",
    "mg spirit mojito":         "other",
    "mg spirit vodka lime":     "other",
    "mg":                       "other",
    "oj":                       "other",
    "revo energy":              "other",
    "revo":                     "other",
    "corelli":                  "other",
    "buzz":                     "other",
    "purple parrot":            "other",
    "oryx":                     "other",
    "oryx sun tap":             "other",
    "greenberg":                "other",
    "malayali":                 "other",
    "kamasutra":                "other",
    "side hustle":              "other",
    "codys":                    "other",
    "cody":                     "other",
    "cody's":                   "other",
    "penguin":                  "other",
    "mio":                      "other",
    "cruise":                   "other",
    "red rose":                 "other",
    "swahill":                  "other",
    "bear":                     "other",
    "wild star":                "other",
    "royal spirit":             "other",
    "majlis":                   "other",
    "beirut":                   "other",
    "chumchurum":               "other",
    "ampm":                     "other",
    "am:pm":                    "other",
    "wignac":                   "other",
    "artevelde":                "other",
    "brewgooder":               "other",
    "black devil":              "other",
    "colt":                     "other",
    "ryders":                   "other",
    "knock out":                "other",
    "66":                       "other",
    "na":                       "other",
    "ok":                       "other",
    "rockberg":                 "other",
    "good of san sebastian":    "other",
    "edelhopfen":               "other",
    "vitalsberg":               "other",
    "starnberger":              "other",

    # fallback
    "other":                    "other",
}

In [ ]:

strict_segments = [
    "super premium", "premium", "mainstream",
    "strong", "craft / speciality", "cider/ flavored", "other"
]

# Map cleaned_brand -> segment
segment_mapping_lower = {k.lower(): v for k, v in segment_mapping_updated.items()}

# Step 1: map
df = df.with_columns(
    pl.col("cleaned_brand")
    .str.to_lowercase()
    .replace(segment_mapping_lower)
    .alias("segment_temp")
)

# Step 2: validate
df = df.with_columns(
    pl.when(pl.col("segment_temp").is_in(strict_segments))
    .then(pl.col("segment_temp"))
    .otherwise(pl.lit("other"))
    .alias("segment")
)

# Step 3: cleanup
df = df.drop("segment_temp")

In [ ]:
df["segment"].unique().to_list()

In [ ]:
df = df.with_columns(
    pl.lit("2026-05-26").alias("date")
)

df

In [ ]:
# drop 2 columns
df = df.drop(["product_html_s3_key", "product_html_s3_uri", "product_source_s3_key", "product_source_s3_uri"])

df

In [ ]:
# -------------------------
# ✅ Reconcile against source of truth before final push
#
# The fields below are LLM-extracted per run and can be wrong. Where the
# same `link` already has a verified row in beer_pricing_analysis.products_source,
# prefer that table's values over this run's freshly extracted ones.
# -------------------------
reconcile_cols = ["cleaned_brand", "sub_type", "platform", "brand", "quantity", "units", "pack_format"]

source_of_truth = db.read_sql(
    f"SELECT link, {', '.join(reconcile_cols)} FROM beer_pricing_analysis.products_source",
    override_db="raw_data",
)
# products_source has link as a primary key, so this join cannot duplicate rows in df
assert source_of_truth["link"].n_unique() == source_of_truth.height

rows_before = df.height
before = df.select(reconcile_cols)

df = df.join(source_of_truth, on="link", how="left", suffix="_truth")
assert df.height == rows_before  # left join on a PK must preserve row count

matched = (
    df.select([f"{c}_truth" for c in reconcile_cols])
    .select(pl.any_horizontal(pl.all().is_not_null()))
    .to_series()
    .sum()
)

for col in reconcile_cols:
    df = df.with_columns(
        pl.coalesce([pl.col(f"{col}_truth"), pl.col(col)]).alias(col)
    )

after = df.select(reconcile_cols)
df = df.drop([f"{col}_truth" for col in reconcile_cols])

# quick visibility into how many rows actually got corrected per column
for col in reconcile_cols:
    changed = (before[col] != after[col]).sum()
    print(f"{col}: {changed} row(s) corrected from source of truth")
print(f"({matched} of {df.height} rows matched a link in products_source)")


In [ ]:
df.write_database(
    table_name="beer_pricing_analysis.plaformsV2",
    connection=db.get_uri(override_db="raw_data"),
    if_table_exists="append"
)